Typical order:

Handle missing values
Detect/manage outliers
Scale/normalize
Encode categoricals
Train model

In [ ]:
import numpy as np
import pandas as pd
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [ ]:
# TODO need to explain all
# TODO add prints after actions (removal, etc)
# TODO define targets, hypothesis, question to answer
# At the end, evaluate predition performance

In [ ]:
"""
Attack vectors show strong patterns based on company size, industry, and employee count
Ransomware and phishing dominate (60%+), with APTs targeting large enterprises
Financial services and healthcare sectors experience distinct attack patterns
Tree-based ML models achieve 70-75% accuracy in attack vector prediction
Feature importance reveals revenue, employee count, and industry as top predictors
"""

In [3]:
# Extract CSV's to PD's dataframes
df_financial = pd.read_csv("data/raw/financial_impact.csv")
df_incidents = pd.read_csv("data/raw/incidents_master.csv")
df_market = pd.read_csv("data/raw/market_impact.csv")

df_merged = df_incidents.merge(df_financial, on='incident_id', how='inner').merge(df_market, on='incident_id', how='inner')

In [4]:
# distinguish between numerical and categorical features
def define_feature_types(df):
    numerical_features = df.select_dtypes(include=['number']).columns
    categorical_features = df.select_dtypes(include=['object', 'string', 'category', 'bool']).columns
    return numerical_features, categorical_features

num_cols, cat_cols = define_feature_types(df_merged)
len(num_cols)+len(cat_cols)==df_merged.shape[1]

True

## 2.1. Data Deletion: Missing Values

In [ ]:
missing_cols = pd.DataFrame({'missing_count': df_merged.isna().sum()})
missing_cols['missing_%'] = (missing_cols['missing_count']/df_merged.shape[0]*100)
missing_cols = missing_cols.sort_values('missing_%', ascending=False).round(2) # TODO Can preview, or should it go in the analysis part exclusively? (otherwise could delete code)

missing_rows = pd.DataFrame({'missing_count': df_merged.isna().sum(axis=1)})
missing_rows['missing_%'] = (missing_rows['missing_count'] / len(num_cols) * 100)
missing_rows = missing_rows.sort_values('missing_%', ascending=False).round(2) # TODO Can preview, or should it go in the analysis part exclusively?

cols_to_drop = missing_cols[missing_cols["missing_%"] > 30].index # Threshold = 30% TODO decide %
rows_to_drop = missing_rows[missing_rows['missing_%'] > 60].index # Threshold = 60%

df_reduced_miss = df_merged.drop(index=rows_to_drop).drop(columns=cols_to_drop)

## 2.2. Data Split

In [ ]:
# Data split (avoid leakage, explain)
# validation set?? size

In [ ]:
target_name = "?"
X = df_reduced_miss.drop(columns=['target_column']) # Features
y = df_reduced_miss['target_column'] # Target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

## 2.3. Data Deletion: Outliers

In [6]:
def get_iqr_bounds(df, cols):
    limits = df[cols].quantile([0.25, 0.75])
    limits.index = ["Q1", "Q3"]
    limits.loc["IQR"]         = limits.loc["Q3"] - limits.loc["Q1"]
    limits.loc["inner_lower"] = limits.loc["Q1"] - 1.5 * limits.loc["IQR"]
    limits.loc["inner_upper"] = limits.loc["Q3"] + 1.5 * limits.loc["IQR"]
    limits.loc["outer_lower"] = limits.loc["Q1"] - 3.0 * limits.loc["IQR"]
    limits.loc["outer_upper"] = limits.loc["Q3"] + 3.0 * limits.loc["IQR"]
    return limits

In [7]:
limits = get_iqr_bounds(df_merged, num_cols) # TODO Could print to be informative

# Potential outliers (inner fence, includes extreme outliers)
outlier_mask_inner = (df_merged[num_cols] < limits.loc["inner_lower"]) | (df_merged[num_cols] > limits.loc["inner_upper"])
# Extreme outliers (outer fence, subset of the previous)
outlier_mask_outer = (df_merged[num_cols] < limits.loc["outer_lower"]) | (df_merged[num_cols] > limits.loc["outer_upper"])

outlier_mask_inner_isolated = outlier_mask_inner & ~outlier_mask_outer # Differentiate the two kind of oultiers for different management


In [ ]:
# Capping/Windsorizing for inner values (replace with fence value), Dropping for outer values 
# a) Drop rows with extreme outliers
rows_to_drop = outlier_mask_outer.any(axis=1) # Feature has at least a True value on whether it is an outlier
df_reduced_out = df_merged[~rows_to_drop]

# b) Cap rest of potential outliers
df_reduced_out[num_cols] = df_merged[num_cols].clip(lower=limits.loc["inner_lower"], upper=limits.loc["inner_upper"], axis=1)


## 2.4. Data Imputation

In [ ]:
imputer = IterativeImputer(estimator=BayesianRidge(), max_iter=10)

X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

## 2.5. Data Scaling

In [ ]:
scaler = StandardScaler()

bool_cols = df_merged[num_cols].select_dtypes(include='bool').columns # Exclude boolean features
binary_cols = [col for col in num_cols if df_merged[col].nunique() == 2] # Exclude binary features
cols_to_exclude = set(bool_cols) | set(binary_cols)
scale_cols = [col for col in num_cols if col not in cols_to_exclude]

df_scaled = df_merged.copy()
df_scaled[scale_cols] = scaler.fit_transform(df_merged[scale_cols])
df_scaled[num_cols].describe()

## 2.6. PCA

In [ ]:
# Need variance % to be between 85-95, reduce up until that pointn

n = len(num_cols)/2
pca = PCA(n_components=n) # TODO choose n, justify
pca.fit(df_scaled[num_cols])


In [ ]:
pca.fit(Xcentred)

print("\nThese are the calculated components: \n", pca.components_) # The PCA object now contains useful information in its attributes. components_ stores the principal components calculated
print("\nThe components we have found, explain the following percentage of variance: \n", pca.explained_variance_ratio_) # explained_variance_ratio_ stores the percentage of variance each of our components explains

# Now apply the PCA transform on the original data
Xnew2 = pca.transform(Xcentred)

print("\nOriginal size of our dataset: ", Xcentred.shape)
print("\nReduced size of our dataset: ", Xnew2.shape)

In [ ]:
# fit with all components first
pca_full = PCA()
pca_full.fit(X_train_scaled)
pca = PCA(n_components=0.95)  # automatically finds n for 95% variance
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)
X_train_pca = pca.fit_transform(X_train_scaled)

## 2.7. Feature Encoding

In [ ]:
# Feature encoding (for example, potential targets must be turned to numerical) -> TODO need potential cat or num targets
cat_cols

Index(['incident_id', 'company_name', 'country_hq', 'industry_primary',
       'industry_secondary', 'is_public_company', 'stock_ticker_x',
       'incident_date', 'incident_date_estimated', 'discovery_date',
       'disclosure_date', 'attack_vector_primary', 'attack_vector_secondary',
       'attack_chain', 'attributed_group', 'attribution_confidence',
       'data_type', 'systems_affected', 'data_source_primary',
       'data_source_secondary', 'data_source_type', 'quality_grade',
       'review_flag', 'notes_x', 'created_at_x', 'updated_at_x',
       'direct_loss_method', 'ransom_source', 'total_loss_method',
       'cpi_index_used', 'notes_y', 'created_at_y', 'updated_at_y',
       'stock_ticker_y', 'sector_index', 'earnings_announcement_within_7d',
       'notes', 'created_at', 'updated_at'],
      dtype='object')

In [14]:
num_cols

Index(['company_revenue_usd', 'employee_count', 'data_compromised_records',
       'downtime_hours', 'confidence_tier', 'quality_score', 'direct_loss_usd',
       'ransom_demanded_usd', 'ransom_paid_usd', 'recovery_cost_usd',
       'legal_fees_usd', 'regulatory_fine_usd', 'insurance_payout_usd',
       'total_loss_usd', 'total_loss_lower_bound', 'total_loss_upper_bound',
       'inflation_adjusted_usd', 'price_7d_before', 'price_disclosure_day',
       'price_1d_after', 'price_7d_after', 'price_30d_after',
       'volume_avg_30d_baseline', 'volume_disclosure_day',
       'sector_return_same_period', 'abnormal_return_1d', 'abnormal_return_7d',
       'abnormal_return_30d', 'car_neg1_to_pos1', 'car_0_to_7', 'car_0_to_30',
       'car_0_to_90', 't_statistic_1d', 'p_value_1d', 't_statistic_30d',
       'p_value_30d', 'market_cap_at_disclosure', 'volume_ratio_disclosure',
       'pre_incident_volatility_30d', 'post_incident_volatility_30d',
       'days_to_price_recovery'],
      dtype='ob